[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Jibby2k1/SPS_Curriculum/blob/main/Intro_Math/Information_Theory/Information_Theory.ipynb)


**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Information Theory

One framework that explains compression limits, communication limits, and — through KL divergence — why cross-entropy is *the* machine learning loss. Four sessions from 'what is a bit?' to the channel coding theorem, with every quantity computed live.

## 0. Introduction

Shannon's move: measure information as **surprise**. Rare events carry more information than expected ones, and $-\log_2 p$ is the unique surprise measure (up to base) that is continuous, decreasing in $p$, and additive over independent events.

## 1. Pre-requisites

[Random Variables](../Analysis/Random_Variables.ipynb) (expectation, distributions); [Independence](../Analysis/Independence.ipynb) for the coding arguments.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(0)

def H(p):
    """Entropy in bits of a probability vector."""
    p = np.asarray(p, float); p = p[p > 0]
    return -(p * np.log2(p)).sum()

---
### 🕐 Session 1 of 4 — *Entropy* (~35 min)
**Goal:** quantify average surprise; see entropy as the compression limit.
**Builds on:** [Random Variables](../Analysis/Random_Variables.ipynb). &nbsp; **Feeds into:** Session 2 (KL & cross-entropy).

---

## 2. Entropy

💡 **Intuition.** Entropy $H(X) = E[-\log_2 p(X)]$ is the **average surprise** of a source — equivalently, the number of yes/no questions you need *on average* to pin down an outcome, when you ask cleverly. A fair coin: 1 bit. A loaded coin: less, because you can exploit the bias. That question-count reading *is* the compression story: you cannot losslessly encode a source below $H$ bits/symbol on average (Shannon's source coding theorem), and Huffman codes get within 1 bit of it.

In [ ]:

# YOUR CODE HERE


**What just happened.** The binary entropy curve: **1 bit** at $p = 0.5$, falling to **zero** at both extremes, and symmetric about the middle.

Read the two endpoints first, because they anchor the whole quantity. At $p = 0$ or $p = 1$ the outcome is *certain*, so observing it tells you nothing and the entropy is exactly zero. At $p = 0.5$ you are maximally uncertain and one yes/no question is exactly what it takes. **Entropy measures uncertainty, and uncertainty is maximised by the uniform distribution** — a fact that generalises to any alphabet, and the reason the maximum-entropy principle picks uniform when you know nothing.

**The symmetry is worth a remark too.** $H(0.9) = H(0.1)$: a coin that is 90% heads is exactly as uncertain as one that is 90% tails. Entropy does not care *which* outcome is likely, only how concentrated the distribution is.

**And note how flat the peak is.** Entropy stays above 0.9 bits across roughly $p \in [0.3, 0.7]$ — so a substantially biased coin is still nearly as unpredictable as a fair one. The payoff from exploiting bias only becomes large once the bias is extreme, which is exactly what the next cell measures: at $p = 0.9$ there are still 0.469 bits per symbol to encode, and only at $p = 0.99$ does the floor drop to 0.081.

**Why $-\log_2 p$ and not some other measure of surprise?** Because three requirements force it: surprise should be continuous in $p$, decreasing (rarer events are more surprising), and **additive over independent events** — learning two independent facts should surprise you by the sum of their surprises. Additivity over independence is what demands a logarithm, and the base merely chooses the unit. Base 2 gives bits; base $e$ gives nats. Entropy is then just the *average* surprise, $E[-\log_2 p(X)]$.

In [ ]:
# Entropy = compression limit, demonstrated with a real compressor (zlib)

# YOUR CODE HERE


**What just happened.** A real compressor measured against a theoretical floor — and the result is more interesting than the printed conclusion suggests:

| $P(1)$ | entropy | zlib | above the floor |
|---|---|---|---|
| 0.5 | 1.000 | 1.001 | +0% |
| 0.9 | 0.469 | 0.565 | **+20%** |
| 0.99 | 0.081 | 0.115 | **+42%** |

**The floor holds — that part is a theorem.** No row goes below its entropy, and none ever could: Shannon's source coding theorem says lossless encoding of an i.i.d. source requires at least $H$ bits per symbol on average, and violating it would mean two different inputs mapping to the same output. The 1.001 at $p = 0.5$ is the honest picture of a compressor facing incompressible data — it cannot help and adds a whisker of overhead.

**But "hugs the floor" overstates the other two rows.** 20% and 42% above optimal is a real gap, and it is worth understanding rather than glossing.

Two causes, and they are different in kind. First, **zlib is the wrong tool**: it is a general-purpose compressor built around byte-level repetition (LZ77 plus Huffman), not an optimal code for a known Bernoulli source. An arithmetic coder handed the true $p$ would come far closer to the floor. Second, and more fundamental, the source coding theorem is an **asymptotic** statement about long blocks. Every practical code carries per-symbol and per-block overhead, and notice that the *absolute* overhead is roughly constant across all three rows at 0.03–0.1 bits/symbol — it is only the *relative* cost that explodes as the entropy shrinks toward zero.

**So separate the two claims, because they are frequently conflated.** "You cannot beat entropy" is a theorem, exact and universal. "Real compressors approach entropy" is an engineering statement that is true for well-matched codecs and long inputs, and visibly false for a mismatched general-purpose tool on a highly skewed source. Students who merge them come away believing compression is solved.

**And the practical reading is still striking.** At $P(1) = 0.99$ the data compresses roughly 70× (0.115 bits per symbol against 8 bits stored raw), which is why skewed data is worth compressing at all — even a 42% inefficiency leaves an enormous win. The entropy tells you the prize; the codec decides how much of it you collect.

---
### 🕐 Session 2 of 4 — *KL Divergence & Cross-Entropy* (~40 min)
**Goal:** measure the cost of believing the wrong distribution; derive the ML loss.
**Builds on:** Session 1. &nbsp; **Feeds into:** Session 3 (mutual information).

---

## 3. Relative Entropy

💡 **Intuition.** Suppose the world emits symbols from $p$ but you built your code (or your model) for $q$. You pay $E_p[-\log_2 q(X)]$ bits per symbol — the **cross-entropy** — instead of the optimal $H(p)$. The overpayment is the KL divergence:
$D(p\|q) = \sum_x p(x) \log \frac{p(x)}{q(x)} \ge 0$ — *the price of wrong beliefs, in bits*. Minimizing cross-entropy in ML is minimizing that price: training a classifier literally optimizes its codebook for the data distribution.

### Proof: $D(p\|q) \ge 0$ (Gibbs' inequality)

Since $\log$ is concave, Jensen's inequality gives
$$-D(p\|q) = \sum_x p(x) \log\frac{q(x)}{p(x)} \le \log \sum_x p(x)\frac{q(x)}{p(x)} = \log \sum_x q(x) = \log 1 = 0,$$
with equality iff $p = q$. $\blacksquare$ Two warnings: $D$ is **not symmetric** and violates the triangle inequality — a directed cost, not a distance.

In [ ]:
# Cross-entropy loss IS log-loss with a KL floor

# YOUR CODE HERE


**What just happened.** Three candidate models scored against the same truth, and every row decomposes the same way:

$$\text{cross-entropy} = \underbrace{H(p)}_{\text{1.157, fixed}} + \underbrace{D(p\|q)}_{\text{0.000 / 0.039 / 1.684}}$$

**The fixed term is the important one.** $H(p) = 1.157$ bits appears in every row and cannot be reduced by any model, because it is the data's *own* uncertainty. Even a perfect model — $q = p$ exactly — pays 1.157 bits per symbol. The only part a model can improve is the KL term.

That has a direct consequence students routinely get wrong: **a nonzero cross-entropy loss is not evidence of a bad model.** A classifier converging to a loss of 1.157 on this data has achieved perfection. If your training loss plateaus above zero, the first question is what $H(p)$ is for your labels — irreducible label noise sets a floor exactly as the noise floor did in [Intro_RNN](../../Intro_Time_Series/Intro_RNN.ipynb)'s forecasting bake-off. Chasing zero loss on noisy labels means overfitting.

**And the KL column is a literal price list.** The "close" model wastes 0.039 bits per symbol; the "wrong" one wastes 1.684 — more than the entire entropy of the source. Multiply by symbols per second and you have the cost of a bad model in bits, which is what KL is: **the price of wrong beliefs**. If you built a code for $q$ and the world emits $p$, that is your overpayment.

**Which is why cross-entropy is *the* machine learning loss, not merely a convenient one.** Since $H(p)$ does not depend on the model parameters, minimising cross-entropy is *exactly* minimising $D(p\|q)$. Training a classifier is optimising its codebook for the data distribution — every classifier you have trained was doing information theory, whether or not it was framed that way.

**Two properties of KL worth carrying, both visible here.** It is non-negative (Gibbs' inequality, via Jensen on the concavity of $\log$), with equality precisely when $q = p$ — which the first row confirms at 0.000. And it is **not symmetric**: $D(p\|q) \neq D(q\|p)$ in general, and it violates the triangle inequality, so it is a *directed cost* rather than a distance.

That asymmetry has real modelling consequences. Minimising $D(p\|q)$ over $q$ — forward KL, what maximum likelihood does — forces $q$ to cover all of $p$'s mass, producing broad, blurry fits. Minimising $D(q\|p)$ — reverse KL, what variational inference does — lets $q$ concentrate on a single mode and ignore the rest. Same two distributions, opposite failure modes, decided entirely by which argument you put first.

---
### 🕐 Session 3 of 4 — *Mutual Information* (~35 min)
**Goal:** quantify what one variable tells you about another; data processing can only lose it.
**Builds on:** Session 2. &nbsp; **Feeds into:** Session 4 (coding at a glance).

---

## 4. Mutual Information

💡 **Intuition.** $I(X;Y) = H(X) - H(X|Y)$: how many bits of uncertainty about $X$ does observing $Y$ remove? Equivalently $I(X;Y) = D(p_{XY} \| p_X p_Y)$ — the KL cost of pretending they're independent. Zero iff independent ([Independence](../Analysis/Independence.ipynb), quantified!), and — unlike correlation — it detects *nonlinear* dependence too.

In [ ]:

# YOUR CODE HERE


**What just happened.** Three pairs of variables, and the third row is the one that matters:

| relationship | correlation | mutual information |
|---|---|---|
| independent | +0.000 | 0.003 bits |
| linear, $y = x + n$ | +0.894 | 1.106 bits |
| **nonlinear, $y = x^2 + n$** | **−0.001** | **0.979 bits** |

**Correlation reports nothing for $y = x^2$, and it is completely wrong to conclude independence.** Knowing $x$ determines $y$ up to noise — the dependence is essentially total — yet the linear measure sees none of it. The reason is mechanical: correlation averages products $xy$, and for a symmetric parabola the positive and negative $x$ contribute equal and opposite products that cancel exactly. Mutual information reports 0.979 bits, nearly as much dependence as the linear case.

This is worth stating as a warning rather than a curiosity: **"uncorrelated" does not mean "independent."** A correlation matrix showing zeros can hide arbitrarily strong nonlinear structure, and a feature discarded for low correlation may be the most informative one you had. Mutual information is zero **iff** independence holds — that is the definition $I(X;Y) = D(p_{XY}\|p_Xp_Y)$, the KL cost of pretending independence, combined with Session 2's Gibbs inequality.

**Now read the "independent" row correctly, because 0.003 is not zero and that is not an accident.** Histogram-based MI estimators are biased **upward**, by roughly
$$\frac{(b_x-1)(b_y-1)}{2N\ln 2} = \frac{23 \times 23}{2 \times 10^5 \times 0.693} = 0.0038 \text{ bits}$$
with the 24×24 bins and $N = 10^5$ used here — which matches the measured 0.003 almost exactly. So the honest reading is "consistent with zero once the known estimator bias is accounted for," not "a small amount of dependence."

**That bias is the practical trap in this session.** It grows with the number of bins and shrinks with sample size, so a naive MI estimate will *always* report some dependence between genuinely independent variables. Use 100 bins instead of 24 and the bias rises to about 0.07 bits — twenty times larger, and easily mistaken for a real signal. Anyone using MI for feature selection needs a null baseline: shuffle one variable and estimate MI again to measure your own estimator's floor, exactly as you would compare a detector against chance.

**And the structural result the next cell states.** The **data processing inequality**: if $X \to Y \to Z$ then $I(X;Z) \le I(X;Y)$. No processing can create information about $X$ — every network layer, filter, and feature extractor can only preserve or destroy it. That makes "garbage in, garbage out" a theorem. The subtlety worth adding is that processing *can* make information more **accessible** without increasing it, which is precisely what representation learning does: same bits, better arranged.

**Data processing inequality** (stated): if $X \to Y \to Z$ is a Markov chain (Z computed from Y alone), then $I(X;Z) \le I(X;Y)$ — **no processing can create information about $X$**. Deep networks, filters, features: every stage can only preserve or destroy. This is the information-theoretic backbone of representation learning, and the reason 'garbage in, garbage out' is a theorem.

---
### 🕐 Session 4 of 4 — *Coding at a Glance* (~35 min)
**Goal:** the two Shannon theorems, one channel capacity computed, and the SNR connection.
**Builds on:** Sessions 1–3.

---

## 5. The Two Theorems

**Source coding** (S1's floor): lossless compression needs $\ge H$ bits/symbol.

**Channel coding.** A noisy channel has capacity $C = \max_{p_X} I(X;Y)$; *any* rate below $C$ is achievable with vanishing error (via long random-ish codes), and no rate above it is. The shocking part: noise does **not** cap reliability — only *rate*.

💡 **Intuition.** Long codewords let the law of large numbers ([Independence](../Analysis/Independence.ipynb)) concentrate the noise: typical received sequences cluster into disjoint balls around codewords, and decoding is 'which ball am I in?'. Capacity counts how many disjoint balls fit.

In [ ]:
# Capacity of the binary symmetric channel: C = 1 − H(ε)

# YOUR CODE HERE


**The formula on every comms slide.** For the Gaussian channel with signal-to-noise ratio SNR: $C = \tfrac12 \log_2(1 + \mathrm{SNR})$ bits per use — bandwidth and SNR are the currency of every link budget in [Digital Communications](../../Intro_DSP/Digital_Communications.ipynb).

## 6. Conclusion

Entropy = surprise = compression floor; KL = the bits you waste believing $q$ when truth is $p$ (cross-entropy loss, demystified); mutual information = dependence in bits, immune to nonlinearity, only ever destroyed by processing; capacity = the rate ceiling noise imposes. Four numbers that govern every pipeline in this curriculum.

---
## Where next

- [Digital Communications](../../Intro_DSP/Digital_Communications.ipynb) — engineering toward capacity.
- [Training Dynamics](../../Intro_Mach_Learn/Training_Dynamics.ipynb) — cross-entropy at work, at scale.
- [Estimation Theory](../Estimation_Theory/Estimation_Theory.ipynb) — Fisher information: KL's local curvature.